# Toxic Comment Classifier - Experimentation Notebook

This notebook allows you to experiment with the toxic comment classifier models.

## 1. Setup and Imports

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append('src')

from preprocessing import TextPreprocessor, load_and_prepare_data
from predict import ToxicCommentPredictor

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Imports successful!")

## 2. Load and Explore Data

In [ ]:
# Load data
data_path = 'data/train.csv'

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"✅ Loaded {len(df)} samples")
    print("\nDataset shape:", df.shape)
    print("\nColumns:", df.columns.tolist())
    df.head()
else:
    print("❌ Dataset not found! Generate it with: python src/generate_sample_data.py")

In [ ]:
# Data statistics
if 'df' in locals():
    print("Dataset Info:")
    print(df.info())
    
    print("\nBasic Statistics:")
    print(df.describe())

In [ ]:
# Class distribution
if 'df' in locals():
    toxic_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
    
    if all(col in df.columns for col in toxic_cols):
        fig, axes = plt.subplots(2, 3, figsize=(15, 8))
        axes = axes.flatten()
        
        for idx, col in enumerate(toxic_cols):
            df[col].value_counts().plot(kind='bar', ax=axes[idx], color=['green', 'red'])
            axes[idx].set_title(f'{col.capitalize()} Distribution')
            axes[idx].set_xlabel('Class')
            axes[idx].set_ylabel('Count')
        
        plt.tight_layout()
        plt.show()

## 3. Text Preprocessing

In [ ]:
# Initialize preprocessor
preprocessor = TextPreprocessor()

# Test preprocessing
test_text = "This is a TEST comment with URLs http://example.com and @mentions #hashtags!!!"
processed = preprocessor.preprocess_text(test_text)

print("Original:", test_text)
print("Processed:", processed)

In [ ]:
# Preprocess sample comments
if 'df' in locals():
    sample_df = df.head(10).copy()
    sample_df['processed'] = sample_df['comment_text'].apply(preprocessor.preprocess_text)
    
    print("Sample Preprocessing Results:")
    for idx, row in sample_df.iterrows():
        print(f"\n{idx+1}. Original: {row['comment_text'][:100]}...")
        print(f"   Processed: {row['processed'][:100]}...")

## 4. Load Trained Models

In [ ]:
# Initialize predictor
predictor = ToxicCommentPredictor(models_dir='models')

# Load models
predictor.load_all_models()

## 5. Make Predictions

In [ ]:
# Test comments
test_comments = [
    "This is a great article, thanks for sharing!",
    "You are an idiot and nobody likes you",
    "I really appreciate your perspective on this topic",
    "This is stupid and so are you",
    "Interesting point of view, I hadn't considered that"
]

# Predict
for i, comment in enumerate(test_comments, 1):
    print(f"\n{'='*60}")
    print(f"Comment {i}: \"{comment}\"")
    print('='*60)
    
    results = predictor.predict_all(comment)
    
    for model_name, result in results.items():
        if 'error' not in result:
            emoji = "🚫" if result['prediction'] == 1 else "✅"
            print(f"{emoji} {model_name:20s}: {result['label']:12s} (confidence: {result['probability']:.2%})")
        else:
            print(f"❌ {model_name}: Error - {result['error']}")

## 6. Interactive Prediction

In [ ]:
# Enter your own comment
user_comment = input("Enter a comment to analyze: ")

if user_comment.strip():
    print(f"\nAnalyzing: \"{user_comment}\"\n")
    
    results = predictor.predict_all(user_comment)
    
    # Display results
    for model_name, result in results.items():
        if 'error' not in result:
            emoji = "🚫" if result['prediction'] == 1 else "✅"
            print(f"{emoji} {model_name:20s}: {result['label']:12s} (confidence: {result['probability']:.2%})")
    
    # Consensus
    predictions = [r['prediction'] for r in results.values() if 'error' not in r]
    if predictions:
        consensus = sum(predictions) / len(predictions)
        print(f"\nConsensus: {consensus:.0%} of models predict TOXIC")
else:
    print("No comment entered.")

## 7. Model Comparison Visualization

In [ ]:
# Load model comparison results
comparison_path = 'models/model_comparison.csv'

if os.path.exists(comparison_path):
    comparison_df = pd.read_csv(comparison_path)
    
    # Display table
    print("Model Comparison:")
    print(comparison_df.to_string(index=False))
    
    # Plot comparison
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(comparison_df))
    width = 0.2
    
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
    
    for i, metric in enumerate(metrics):
        ax.bar(x + i*width, comparison_df[metric], width, label=metric, color=colors[i])
    
    ax.set_xlabel('Models', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(comparison_df['Model'])
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1.1])
    
    plt.tight_layout()
    plt.show()
else:
    print("Model comparison file not found. Train models first.")

## 8. Batch Prediction

In [ ]:
# Batch prediction on sample data
if 'df' in locals():
    sample_size = 20
    sample_df = df.sample(n=min(sample_size, len(df)), random_state=42)
    
    # Predict with LSTM (fastest for batch)
    if predictor.lstm_loaded:
        predictions = []
        
        for idx, row in sample_df.iterrows():
            pred, prob = predictor.predict_lstm(row['comment_text'])
            predictions.append({
                'comment': row['comment_text'][:50] + '...',
                'true_label': row.get('toxic', 'Unknown'),
                'predicted': pred,
                'probability': prob
            })
        
        pred_df = pd.DataFrame(predictions)
        print("Batch Prediction Results:")
        print(pred_df.to_string(index=False))
    else:
        print("LSTM model not loaded")

## 9. Error Analysis

In [ ]:
# Analyze prediction errors
if 'pred_df' in locals() and 'true_label' in pred_df.columns:
    # Calculate accuracy
    correct = (pred_df['true_label'] == pred_df['predicted']).sum()
    total = len(pred_df)
    accuracy = correct / total
    
    print(f"Accuracy: {accuracy:.2%} ({correct}/{total})")
    
    # Show errors
    errors = pred_df[pred_df['true_label'] != pred_df['predicted']]
    
    if len(errors) > 0:
        print(f"\nErrors ({len(errors)}):")
        print(errors.to_string(index=False))
    else:
        print("\n✅ No errors in this sample!")

## 10. Custom Experiments

Use this section for your own experiments!

In [ ]:
# Your code here
